# Step 03 — OSM POI Cleaning & Consolidation

**Input:** `data/output/01_all_pois.gpkg`  
**Output:** `data/output/03_osm_pois_modified.gpkg`

**Steps:**
1. Drop irrelevant columns
2. Merge address fields into a single `addr:full` column
3. Fill `name` from `operator` where missing
4. Build `tags_search` (tokenised, searchable tag text)
5. Build `additional_information` (semicolon-separated remaining fields)
6. Save cleaned POI file

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import pandas as pd
import numpy as np
import ast
import re

pd.set_option('display.max_columns', None)
print('Config loaded')

Config loaded


## 1. Load and drop irrelevant columns

In [ ]:
pois = gpd.read_file(ALL_POIS_FILE, layer='pois')
print(f'Loaded {len(pois):,} POIs')

cols_to_drop = [
    'lat', 'visible', 'timestamp', 'lon', 'version', 'changeset',
    'addr:country', 'addr:full', 'opening_hours', 'phone', 'ref',
    'source', 'start_date', 'building:levels', 'osm_type',
]
pois = pois.drop(columns=[c for c in cols_to_drop if c in pois.columns])
print(f'Remaining columns: {len(pois.columns)}')

Loaded 86,542 POIs
Remaining columns: 87


## 2. Merge address fields

In [ ]:
addr_cols = ['addr:housename', 'addr:street', 'addr:housenumber',
             'addr:place', 'addr:city', 'addr:postcode']

def merge_address(row):
    parts = [
        str(row[c]).strip()
        for c in addr_cols
        if c in row and pd.notna(row[c]) and str(row[c]).strip() not in ('', 'nan')
    ]
    return np.nan if not parts else ', '.join(parts)

pois['addr:full'] = pois.apply(merge_address, axis=1)
pois = pois.drop(columns=[c for c in addr_cols if c in pois.columns])

## 3. Fill name from operator; merge website from url

In [ ]:
if 'operator' in pois.columns:
    pois['name'] = pois['name'].fillna(pois['operator'])
    pois = pois.drop(columns=['operator'])

if 'url' in pois.columns:
    if 'website' in pois.columns:
        pois['website'] = pois['website'].replace('', np.nan).fillna(pois['url'])
    else:
        pois['website'] = pois['url']
    pois = pois.drop(columns=['url'])

## 4. Build tags_search (tokenised, searchable)

In [ ]:
def ensure_dict(x):
    if isinstance(x, dict): return x
    if isinstance(x, str) and x.strip():
        try:
            v = ast.literal_eval(x)
            return v if isinstance(v, dict) else {}
        except Exception:
            return {}
    return {}

_non_alnum = re.compile(r'[^0-9a-zA-ZäöüÄÖÜß]+')
_ws        = re.compile(r'\s+')

def tokenize_text(s):
    s = str(s).lower()
    s = _non_alnum.sub(' ', s)
    return _ws.sub(' ', s).strip()

def tags_to_searchable(d):
    if not isinstance(d, dict) or not d: return np.nan
    raw = ' '.join([f'{k} {v}' for k, v in d.items() if pd.notna(v)])
    out = tokenize_text(raw)
    return np.nan if out == '' else out

if 'tags' in pois.columns:
    pois['tags_dict']   = pois['tags'].apply(ensure_dict)
    pois['tags_search'] = pois['tags_dict'].apply(tags_to_searchable)
    pois = pois.drop(columns=['tags', 'tags_dict'])
else:
    pois['tags_search'] = np.nan

## 5. Build additional_information (remaining columns)

In [ ]:
base_cols = ['id', 'email', 'name', 'website', 'amenity', 'building',
             'shop', 'tourism', 'information', 'geometry', 'addr:full', 'tags_search']

merge_cols = [c for c in pois.columns if c not in base_cols]

def consolidate_info(row):
    parts = []
    for col in merge_cols:
        val = row[col]
        if pd.notna(val):
            parts.append(f'{col}: {val}')
    return '; '.join(parts) if parts else None

pois_clean = pois[base_cols].copy()
pois_clean['additional_information'] = pois.apply(consolidate_info, axis=1)
print(f'Final POI columns: {list(pois_clean.columns)}')

Final POI columns: ['id', 'email', 'name', 'website', 'amenity', 'building', 'shop', 'tourism', 'information', 'geometry', 'addr:full', 'tags_search', 'additional_information']


## 6. Save

In [ ]:
pois_clean.to_file(OSM_POIS_MODIFIED_FILE, driver='GPKG')
print(f'Saved {len(pois_clean):,} POIs → {OSM_POIS_MODIFIED_FILE}')

Saved 86,542 POIs → C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline\capacity-pipeline-generic\data\output\03_osm_pois_modified.gpkg
